# 📈 Price Elasticity Analysis
In this notebook, we analyze the price elasticity of demand for different product categories.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm

from src.utils.config import *
from src.data.data_pipeline import DataPipeline
from src.data.feature_engineering import FeatureEngineer
from src.models.elasticity_estimator import ElasticityEstimator
from src.visualization.plots import plot_elasticity_comparison

## 1. Load & Prepare Data
Load the raw data, clean it, and perform feature engineering.

In [ ]:
df_raw = pd.read_csv(RAW_DATA_DIR / 'sales_data.csv')
pipeline = DataPipeline()
df_clean = pipeline.clean_data(df_raw)
fe = FeatureEngineer()
df = fe.create_features(df_clean)
df.head()

## 2. The Log-Log Model
We estimate elasticity using a log-log regression model:

$$ \ln(Q) = \alpha - \beta \cdot \ln(P) + \gamma X + \varepsilon $$

Where:
- $Q$ is demand (units sold)
- $P$ is price
- $\beta$ is the price elasticity of demand (typically expected to be negative)
- $X$ represents other control variables (e.g., seasonality, promotions)

A $\beta$ of -1.5 means a 1% increase in price leads to a 1.5% decrease in demand.

## 3. Fit Overall Elasticity
Estimate the average elasticity across all products.

In [ ]:
estimator = ElasticityEstimator()
estimator.fit(df)

overall_elasticity = estimator.get_elasticity()
print(f"Overall Elasticity: {overall_elasticity:.3f}")
print("\nModel Summary:")
print(estimator.get_summary())

## 4. Log-Log Scatter Plot
Visualize the relationship between log(price) and log(demand).

In [ ]:
fig = px.scatter(df, x='log_price', y='log_demand', opacity=0.3, 
                 trendline='ols', title='Log-Log Scatter: Price vs Demand')
fig.show()

## 5. Category-Level Elasticity
Different categories may have different price sensitivities.

In [ ]:
category_betas = estimator.fit_by_category(df)
beta_df = pd.DataFrame(list(category_betas.items()), columns=['Category', 'Elasticity'])
beta_df.sort_values(by='Elasticity', ascending=True, inplace=True)
beta_df

## 6. Elasticity Comparison Chart
Visual comparison of elasticity across categories.

In [ ]:
fig = plot_elasticity_comparison(category_betas, title="Price Elasticity by Category")
fig.show()

## 7. Interpretation
- Categories with elasticity < -1 are **elastic** (price sensitive). Lowering prices might increase total revenue.
- Categories with elasticity between -1 and 0 are **inelastic** (less price sensitive). Raising prices could increase total revenue.
- These insights should guide our pricing strategy for each product category.

## 8. Elasticity Over Time
We can estimate elasticity over rolling windows to see if price sensitivity is stable.

In [ ]:
df_sorted = df.sort_values('date').copy()
window = 90
rolling_elasticity = []
dates = []

for i in range(window, len(df_sorted), 30):  # Step by 30 days
    window_df = df_sorted.iloc[i-window:i]
    if len(window_df) >= window:
        temp_est = ElasticityEstimator()
        try:
            temp_est.fit(window_df)
            rolling_elasticity.append(temp_est.get_elasticity())
            dates.append(window_df['date'].iloc[-1])
        except:
            pass

rolling_df = pd.DataFrame({'Date': pd.to_datetime(dates), 'Elasticity': rolling_elasticity})
px.line(rolling_df, x='Date', y='Elasticity', title="90-Day Rolling Elasticity Over Time")

## 9. Save Model
Save the overall elasticity model for future use.

In [ ]:
estimator.save(MODELS_DIR / 'elasticity_model.pkl')
print("Elasticity model saved successfully.")